# 07. Stronger training recipe for the from-scratch Bi-Mamba (PlantDoc)

The 150-epoch baseline plateaued at about 33.7% test accuracy, and SSL pretraining did not move it. With only about 2,100 images, the model overfits (training loss keeps falling while validation stalls). This notebook keeps the **same model, splits and seed** and changes only the training recipe:

- **Mixup / CutMix** (images are blended or patched together, with soft labels)
- **RandAugment** and random erasing, plus the usual crops and flips
- **Weight averaging (EMA)**: validation and test use the averaged weights
- Label smoothing 0.1 (same as before). Optional class-balanced sampling (`BALANCED`)

**I do not know if this will help.** It is the standard remedy for small-data overfitting, and I would guess a gain of a few points, but it can also do nothing. Compare it with the 150-epoch baseline (33.73%), and treat a difference under about 3 points as noise (one seed, 252 test images).

**How to use (important):**
1. Keep `QUICK = True` and run all cells interactively. It trains 2 epochs (about 5 minutes) and checks that every part runs, including the final test step.
2. If it works, set `QUICK = False` and use **Save Version, Save and Run All (Commit)**. Expect about 4 hours.
3. Before committing: Accelerator = GPU T4, Internet = ON, both plant datasets attached.

Results file: `results_strong_ep150.csv`.

## 1. Install (Internet must be ON)

In [ ]:
!pip install -q mambapy

## 2. Settings

In [ ]:
import os, glob, time, math, random, copy, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score

QUICK = True      # FIRST run with True (about 5 minutes) to check the code. Then set False for the real ~4 hour run.
EPOCHS = 2 if QUICK else 150
WARMUP = 1 if QUICK else 10
BATCH  = 16       # lower to 8 if you get an out-of-memory error
LR, WD = 5e-4, 0.05
DIM, DEPTH = 192, 8        # same model as the baseline

# --- the stronger recipe ---
MIX_PROB, MIXUP_ALPHA, CUTMIX_ALPHA, SWITCH = 0.8, 0.8, 1.0, 0.5   # Mixup / CutMix
SMOOTH = 0.1               # label smoothing (the baseline already used 0.1)
EMA_DECAY = 0.998          # weight averaging; validation and test use the averaged weights
BALANCED = False           # True = class-balanced sampling (may help rare classes, may lower accuracy)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEV == "cuda"
torch.backends.cudnn.benchmark = True
SUFFIX = "_quick" if QUICK else "_ep150"
RESULTS = f"/kaggle/working/results_strong{SUFFIX}.csv"
print("device:", DEV, "| epochs:", EPOCHS, "| results file:", RESULTS)

## 3. Data (same code and seed as before, so the splits are identical)

In [ ]:
import os, random, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

SEED = 42
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
PD_ROOT = os.environ.get("PD_ROOT", "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset")
CT_ROOT = os.environ.get("CT_ROOT", "/kaggle/input/datasets/myprojectdictionary/citrus-leaf-disease-image/Citrus Leaf Disease Image")
OUT = os.environ.get("SPLIT_OUT", "/kaggle/working/splits")
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [ ]:
def list_images(root):
    rows = []
    for c in sorted(os.listdir(root)):
        p = os.path.join(root, c)
        if not os.path.isdir(p):
            continue
        for f in sorted(os.listdir(p)):
            if f.lower().endswith(IMG_EXT):
                rows.append((os.path.join(p, f), c))
    return pd.DataFrame(rows, columns=["path", "label"])

def safe_split(df, test_size, seed):
    """Stratified split. Classes with fewer than 2 images cannot be stratified, so they stay in the first part."""
    counts = df.label.value_counts()
    rare = counts[counts < 2].index
    df_rare, df_ok = df[df.label.isin(rare)], df[~df.label.isin(rare)]
    a, b = train_test_split(df_ok, test_size=test_size, stratify=df_ok.label, random_state=seed)
    return pd.concat([a, df_rare]).reset_index(drop=True), b.reset_index(drop=True)

_hash_cache = {}
def file_md5(path):
    if path not in _hash_cache:
        with open(path, "rb") as f:
            _hash_cache[path] = hashlib.md5(f.read()).hexdigest()
    return _hash_cache[path]

def drop_leaks(train, val, test):
    """Remove images from train/val that are byte-identical to a test image (and from train if identical to a val image).
    The test set itself is never changed."""
    test_h = {file_md5(p) for p in test.path}
    val = val[~val.path.map(file_md5).isin(test_h)]
    val_h = {file_md5(p) for p in val.path}
    train = train[~train.path.map(file_md5).isin(test_h | val_h)]
    return train.reset_index(drop=True), val.reset_index(drop=True)

# ---------- PlantDoc: use the provided test set; split train 80/20 into train/val ----------
pd_train_full = list_images(f"{PD_ROOT}/train")
pd_test = list_images(f"{PD_ROOT}/test").reset_index(drop=True)
pd_train, pd_val = safe_split(pd_train_full, 0.20, SEED)
n0 = (len(pd_train), len(pd_val))
pd_train, pd_val = drop_leaks(pd_train, pd_val, pd_test)
print(f"PlantDoc: removed {n0[0]-len(pd_train)} train and {n0[1]-len(pd_val)} val images that duplicate a val/test image")
pd_classes = sorted(pd_train_full.label.unique())          # 28 classes (from train)
pd_map = {c: i for i, c in enumerate(pd_classes)}
unknown = set(pd_test.label) - set(pd_map)
assert not unknown, f"test classes missing from train: {unknown}"

pd_train.to_csv(f"{OUT}/plantdoc_train.csv", index=False)
pd_val.to_csv(f"{OUT}/plantdoc_val.csv", index=False)
pd_test.to_csv(f"{OUT}/plantdoc_test.csv", index=False)
print(f"PlantDoc: train {len(pd_train)} | val {len(pd_val)} | test {len(pd_test)} | classes {len(pd_classes)}")

# ---------- Citrus: no provided split, so 5-fold stratified CV (test = 1 fold, val = 12.5% of the rest) ----------
ct = list_images(CT_ROOT)
ct_classes = sorted(ct.label.unique())
ct_map = {c: i for i, c in enumerate(ct_classes)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ct_folds = []
for k, (tr_idx, te_idx) in enumerate(skf.split(ct, ct.label)):
    rest, test = ct.iloc[tr_idx], ct.iloc[te_idx].reset_index(drop=True)
    train, val = safe_split(rest, 0.125, SEED)
    train, val = drop_leaks(train, val, test)
    ct_folds.append((train, val, test))
    train.to_csv(f"{OUT}/citrus_fold{k}_train.csv", index=False)
    val.to_csv(f"{OUT}/citrus_fold{k}_val.csv", index=False)
    test.to_csv(f"{OUT}/citrus_fold{k}_test.csv", index=False)
    assert not (set(train.path) & set(test.path)) and not (set(val.path) & set(test.path)) and not (set(train.path) & set(val.path))
print(f"Citrus: {len(ct)} images, classes {ct_classes}")
for k, (tr, va, te) in enumerate(ct_folds):
    print(f"  fold {k}: train {len(tr)} | val {len(va)} | test {len(te)}  | melanose in test: {(te.label == 'Melanose').sum()}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.4, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LeafDS(Dataset):
    def __init__(self, df, cmap, tf):
        self.paths = df.path.tolist()
        self.y = [cmap[l] for l in df.label]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.y[i]

def make_loaders(train_df, val_df, test_df, cmap, bs=32, workers=2, balanced=False):
    tr = LeafDS(train_df, cmap, train_tf)
    sampler = None
    if balanced:   # class-balanced sampling (the paper reports this setting in Table 11)
        cnt = np.bincount(tr.y, minlength=len(cmap)).astype(float)
        w = [1.0 / max(cnt[y], 1) for y in tr.y]
        sampler = WeightedRandomSampler(w, num_samples=len(tr), replacement=True)
    kw = dict(num_workers=workers, pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, bs, shuffle=sampler is None, sampler=sampler, drop_last=True, **kw),
            DataLoader(LeafDS(val_df, cmap, eval_tf), bs, shuffle=False, **kw),
            DataLoader(LeafDS(test_df, cmap, eval_tf), bs, shuffle=False, **kw))

## 4. Encoder (same as the baseline)

In [ ]:
from mambapy.mamba import MambaBlock, MambaConfig

def make_block(d):
    return MambaBlock(MambaConfig(d_model=d, n_layers=1, d_state=16, expand_factor=2,
                                  d_conv=4, pscan=True, use_cuda=False))

class BiMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fwd, self.bwd = make_block(d), make_block(d)
        self.wf = nn.Parameter(torch.tensor(0.5))   # learnable fusion weights (paper Eq. 7)
        self.wb = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        h = self.norm(x)
        return x + self.wf * self.fwd(h) + self.wb * self.bwd(h.flip(1)).flip(1)

class MiniBiMamba(nn.Module):
    def __init__(self, d=192, depth=8, patch=16, base_grid=14):
        super().__init__()
        self.embed = nn.Conv2d(3, d, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
    def forward(self, img):
        t = self.embed(img)
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t), (gh, gw)

class Classifier(nn.Module):
    def __init__(self, n_classes, d=DIM, depth=DEPTH):
        super().__init__()
        self.enc = MiniBiMamba(d, depth)
        self.head = nn.Linear(d, n_classes)
    def forward(self, x):
        tokens, _ = self.enc(x)
        return self.head(tokens.mean(1))

_m = Classifier(28)
print("Parameters (M):", round(sum(p.numel() for p in _m.parameters()) / 1e6, 2)); del _m

## 5. Baseline evaluation helpers

In [ ]:
def evaluate(model, loader, n_classes):
    model.eval(); ys, ps, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss_sum += F.cross_entropy(out.float(), y, reduction="sum").item()
            ps.append(out.argmax(1).cpu()); ys.append(y.cpu()); n += len(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    f1s = f1_score(y, p, average=None, labels=list(range(n_classes)), zero_division=0)
    present = np.unique(y)                      # PlantDoc test has one class fewer than train
    return dict(loss=loss_sum / n, acc=float((y == p).mean() * 100),
                macro_f1=float(f1s[present].mean() * 100), f1s=f1s, present=present)

def minority_f1(res, train_df, cmap):
    """Mean F1 of the 25% rarest classes (by training count) that appear in the test set."""
    cnt = train_df.label.map(cmap).value_counts()
    thr = cnt.quantile(0.25)
    minority = [c for c in res["present"] if cnt.get(c, 0) <= thr]
    return float(res["f1s"][minority].mean() * 100) if minority else float("nan")

def get_split(name, fold):
    if name == "plantdoc":
        return pd_train, pd_val, pd_test, pd_map
    tr, va, te = ct_folds[fold]
    return tr, va, te, ct_map

def already_done(tag):
    return os.path.exists(RESULTS) and tag in set(pd.read_csv(RESULTS)["tag"])

def run_experiment(name, fold=0, seed=0):
    tag = f"{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH)
    n_cls = len(cmap)
    model = Classifier(n_cls).to(DEV)

    decay = [p for n, p in model.named_parameters() if p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n]
    no_decay = [p for n, p in model.named_parameters() if not (p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n)]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WD}, {"params": no_decay, "weight_decay": 0.0}], lr=LR)
    total, warm = EPOCHS * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ck = f"/kaggle/working/ckpt_baseline{SUFFIX}_{tag}.pt"
    start, best, hist = 0, dict(val_acc=-1.0, epoch=-1, state=None), []
    if os.path.exists(ck):
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        scaler.load_state_dict(s["scaler"]); start, best, hist = s["epoch"], s["best"], s["hist"]
        print(f"resumed {tag} from epoch {start}")

    t_start = time.time()
    for ep in range(start, EPOCHS):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue                 # skip a bad step instead of crashing
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1
        r = evaluate(model, lva, n_cls)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), val_loss=r["loss"], val_acc=r["acc"]))
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep + 1, best=best, hist=hist), ck + ".tmp")
        os.replace(ck + ".tmp", ck)
        print(f"[{tag}] epoch {ep+1}/{EPOCHS} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")

    model.load_state_dict(best["state"])                 # test ONLY with the best-validation weights
    res = evaluate(model, lte, n_cls)
    row = dict(tag=tag, dataset=name, fold=fold, seed=seed, epochs=EPOCHS, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
    if os.path.exists(ck): os.remove(ck)                # finished: free the space

## 6. The stronger recipe

In [ ]:
# stronger image augmentation (replaces the baseline's training transform)
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.3, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandAugment(num_ops=2, magnitude=9),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25)])

def mix_batch(x, y, n_cls):
    """Mixup or CutMix with label smoothing. Returns the mixed images and soft targets."""
    onehot = F.one_hot(y, n_cls).float() * (1 - SMOOTH) + SMOOTH / n_cls
    if random.random() > MIX_PROB:
        return x, onehot
    perm = torch.randperm(x.size(0), device=x.device)
    if random.random() < SWITCH:                                   # CutMix
        lam = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA))
        H, W = x.shape[-2:]
        rh, rw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy - rh // 2, 0), min(cy + rh // 2, H)
        x1, x2 = max(cx - rw // 2, 0), min(cx + rw // 2, W)
        x = x.clone()
        x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)                  # exact area that was kept
    else:                                                          # Mixup
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
        x = lam * x + (1 - lam) * x[perm]
    return x, lam * onehot + (1 - lam) * onehot[perm]

def run_strong(name, fold=0, seed=0, label="strong"):
    tag = f"{label}_{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH, balanced=BALANCED)
    n_cls = len(cmap)
    model = Classifier(n_cls).to(DEV)
    ema = copy.deepcopy(model)
    for p in ema.parameters():
        p.requires_grad = False

    def is_decay(n, p):
        return p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n
    decay = [p for n, p in model.named_parameters() if is_decay(n, p)]
    no_decay = [p for n, p in model.named_parameters() if not is_decay(n, p)]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WD}, {"params": no_decay, "weight_decay": 0.0}], lr=LR)
    total, warm = EPOCHS * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ck = f"/kaggle/working/ckpt_strong{SUFFIX}_{tag}.pt"
    start, step, best, hist = 0, 0, dict(val_acc=-1.0, epoch=-1, state=None), []
    if os.path.exists(ck):
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); ema.load_state_dict(s["ema"]); opt.load_state_dict(s["opt"])
        sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
        start, step, best, hist = s["epoch"], s["step"], s["best"], s["hist"]
        print(f"resumed {tag} from epoch {start}")

    t_start = time.time()
    for ep in range(start, EPOCHS):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            x, soft = mix_batch(x, y, n_cls)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = -(soft * F.log_softmax(out.float(), dim=-1)).sum(-1).mean()
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            d = min(EMA_DECAY, (1 + step) / (10 + step))
            with torch.no_grad():
                for pe, p in zip(ema.parameters(), model.parameters()):
                    pe.mul_(d).add_(p.detach(), alpha=1 - d)
            step += 1; tl += loss.item(); nb += 1
        r = evaluate(ema, lva, n_cls)                                      # validate the averaged weights
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), val_loss=r["loss"], val_acc=r["acc"]))
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in ema.state_dict().items()})
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep + 1, step=step, best=best, hist=hist), ck + ".tmp")
        os.replace(ck + ".tmp", ck)
        print(f"[{tag}] epoch {ep+1}/{EPOCHS} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")

    ema.load_state_dict(best["state"])                                     # test ONLY with the best-validation weights
    res = evaluate(ema, lte, n_cls)
    row = dict(tag=tag, label=label, dataset=name, fold=fold, seed=seed, epochs=EPOCHS, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
    if os.path.exists(ck): os.remove(ck)

## 7. The one run

In [ ]:
run_strong("plantdoc", fold=0, seed=0)

## 8. Result

In [ ]:
df = pd.read_csv(RESULTS)
print(df.round(2).to_string(index=False))
print()
print("For comparison (your earlier PlantDoc seed-0 runs):")
print("  baseline, 50 epochs   : test 28.97 | macro-F1 22.86 | minority-F1  3.03")
print("  baseline, 150 epochs  : test 33.73 | macro-F1 30.92 | minority-F1 17.53")
print("  SSL encoder, 150 ep   : test 33.33 | macro-F1 30.08 | minority-F1 20.69")